In [1]:
!pip install -q paddlepaddle paddleocr==2.9.1 opencv-python-headless

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.54.1 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.11.1 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
dm-tree 0.1.10 requires numpy>=2.1.0; python_version >= "3.13", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.11.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
geopandas 1.2.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
ml-dtypes 0.6.0 requires n

In [21]:
import re
from datetime import datetime, timedelta, timezone
import cv2
import numpy as np
from google.colab import files
from paddleocr import PaddleOCR

# 1. Initialize PaddleOCR (angle_cls=False prevents inverted logo hallucinations like MBSJ -> "Esaw")
ocr = PaddleOCR(
    use_angle_cls=False,
    lang="en",
    det_db_thresh=0.22,
    det_db_box_thresh=0.40,
    det_db_unclip_ratio=1.8,
    show_log=False,
)

MYT = timezone(timedelta(hours=8))  # Asia/Kuala_Lumpur (UTC+8)

MALAY_MONTH_MAP = {
    "januari": "01", "january": "01", "jan": "01",
    "februari": "02", "february": "02", "feb": "02",
    "mac": "03", "march": "03", "mar": "03",
    "april": "04", "apr": "04",
    "mei": "05", "may": "05",
    "jun": "06", "june": "06",
    "julai": "07", "july": "07", "jul": "07",
    "ogos": "08", "august": "08", "aug": "08",
    "september": "09", "sep": "09",
    "oktober": "10", "october": "10", "okt": "10", "oct": "10",
    "november": "11", "nov": "11",
    "disember": "12", "december": "12", "dis": "12", "dec": "12",
}

MONTHS = (
    r"(?:Jan(?:uari|uary)?|Feb(?:ruari|ruary)?|Mar(?:ch)?|Mac|Apr(?:il)?|"
    r"May|Mei|Jun(?:e)?|Jul(?:y|ai)?|Aug(?:ust)?|Ogos|Sep(?:tember)?|"
    r"Oct(?:ober)?|Okt(?:ober)?|Nov(?:ember)?|Dec(?:ember)?|Dis(?:ember)?)"
)
DATE_RE = re.compile(
    rf"(?<!\d)(?:\d{{1,2}}[/.-]\d{{1,2}}[/.-]\d{{2,4}}|\d{{4}}[/.-]\d{{1,2}}[/.-]\d{{1,2}}|\d{{1,2}}\s*{MONTHS}\s*\d{{2,4}})",
    re.IGNORECASE,
)
TIME_RE = re.compile(r"(?<!\d)(\d{1,2}[:.]\d{2}(?:[:.,]\d{2})?)\s*(am|pm|AM|PM)?(?!\d)")
POSTCODE_RE = re.compile(r"(?<!\d)(\d{5})(?!\d)")
DAY_RE = re.compile(
    r"\b(?:Monday|Mon|Tuesday|Tue|Wednesday|Wed|Thursday|Thu|Friday|Fri|Saturday|Sat|Sunday|Sun|"
    r"Isnin|Selasa|Rabu|Khamis|Jumaat|Sabtu|Ahad)\b",
    re.IGNORECASE,
)
GPS_RE = re.compile(r"(-?\d{1,3}\.\d+)\s*°?\s*([NS])[,\s]+(-?\d{1,3}\.\d+)\s*°?\s*([EW])", re.IGNORECASE)


# 2. Image Enhancement & OCR Helper
def preprocess_and_ocr(crop_img, darken_concrete: bool = False) -> list[dict]:
    work = crop_img.copy()
    if darken_concrete:
        lut = np.array([((i / 255.0) ** 1.35) * 255 for i in range(256)], dtype="uint8")
        work = cv2.LUT(work, lut)

    upscaled = cv2.resize(work, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
    lab = cv2.cvtColor(upscaled, cv2.COLOR_BGR2LAB)
    l_ch, a_ch, b_ch = cv2.split(lab)
    l_ch = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(l_ch)
    enhanced = cv2.cvtColor(cv2.merge([l_ch, a_ch, b_ch]), cv2.COLOR_LAB2BGR)
    padded = cv2.copyMakeBorder(enhanced, 40, 40, 40, 40, cv2.BORDER_CONSTANT, value=(0, 0, 0))

    result = ocr.ocr(padded, cls=False)
    boxes = []
    if result and result[0]:
        for box, (text, conf) in result[0]:
            if conf >= 0.50:
                boxes.append({
                    "text": text.strip(),
                    "x_left": min(pt[0] for pt in box),
                    "y_top": min(pt[1] for pt in box),
                    "y_bottom": max(pt[1] for pt in box),
                })
    boxes.sort(key=lambda d: (round(d["y_top"] / 25), d["x_left"]))
    return boxes


# 3. Timestamp Normalizer (Outputs DD/MM/YYYY + 24h HH:MM:SS + ISO-8601 TIMESTAMPTZ)
def normalize_timestamp(raw_date: str, raw_time: str) -> dict:
    # Splits both "12Januari" -> "12 Januari" AND "Januari2026" / "Jan2026" -> "Januari 2026" / "Jan 2026"
    clean_d = re.sub(r"(?<=\d)(?=[A-Za-z])|(?<=[A-Za-z])(?=\d)", " ", raw_date.strip())
    dd_mm_yyyy = None

    num_m = re.match(r"^(\d{1,2})[/.-](\d{1,2})[/.-](\d{2,4})$", clean_d)
    if num_m:
        d, m, y = num_m.groups()
        y = f"20{y}" if len(y) == 2 else y
        dd_mm_yyyy = f"{int(d):02d}/{int(m):02d}/{y}"

    txt_m = re.match(r"^(\d{1,2})\s+([A-Za-z]+)\s+(\d{2,4})$", clean_d)
    if txt_m:
        d, mon_str, y = txt_m.groups()
        mon_num = MALAY_MONTH_MAP.get(mon_str.lower())
        y = f"20{y}" if len(y) == 2 else y
        if mon_num:
            dd_mm_yyyy = f"{int(d):02d}/{mon_num}/{y}"

    if not dd_mm_yyyy:
        return {"display_date": "INVALID", "display_time": raw_time, "captured_at_iso": None}

    t_str = raw_time.strip().replace(".", ":")
    t_m = re.match(r"^(\d{1,2}):(\d{2})(?::(\d{2}))?\s*(am|pm)?$", t_str, re.IGNORECASE)
    if t_m:
        hh, mm, ss, meridiem = t_m.groups()
        hh, mm, ss = int(hh), int(mm), int(ss or 0)
        if meridiem:
            if meridiem.lower() == "pm" and hh != 12:
                hh += 12
            elif meridiem.lower() == "am" and hh == 12:
                hh = 0
        dt = datetime.strptime(f"{dd_mm_yyyy} {hh:02d}:{mm:02d}:{ss:02d}", "%d/%m/%Y %H:%M:%S").replace(tzinfo=MYT)
    else:
        dt = datetime.strptime(dd_mm_yyyy, "%d/%m/%Y").replace(tzinfo=MYT)

    return {
        "display_date": dt.strftime("%d/%m/%Y"),
        "display_time": dt.strftime("%H:%M:%S"),
        "captured_at_iso": dt.isoformat(),
    }


# 4. Field Parser & Malaysian Address Normalizer
def finalize_fields(lines: list[str], gps_coords: str = "NOT PRESENT") -> dict:
    # Split fused year+time strings like "9 Jan202608.57.53" -> "9 Jan2026 08.57.53"
    normalized_lines = [re.sub(r"(202\d)(\d{1,2}[:.]\d{2})", r"\1 \2", ln) for ln in lines]
    joined = " ".join(normalized_lines)

    date_m = DATE_RE.search(joined)
    time_m = TIME_RE.search(joined)
    post_m = POSTCODE_RE.search(joined)

    date_str = re.sub(r"(\d)([A-Za-z])", r"\1 \2", date_m.group(0)) if date_m else "NOT FOUND"
    meridiem = time_m.group(2) if (time_m and time_m.group(2)) else ""
    if time_m and not meridiem:
        mer_m = re.search(r"(?<![A-Za-z])(am|pm)(?![A-Za-z])", joined, re.IGNORECASE)
        meridiem = mer_m.group(1) if mer_m else ""
    time_str = f"{time_m.group(1)} {meridiem}".strip() if time_m else "NOT FOUND"
    postcode = post_m.group(1) if post_m else "NOT PRESENT"

    ts = normalize_timestamp(date_str, time_str) if date_str != "NOT FOUND" else {
        "display_date": "NOT FOUND",
        "display_time": time_str,
        "captured_at_iso": None,
    }

    addr_parts = []
    for ln in normalized_lines:
        cleaned = DATE_RE.sub("", ln)
        cleaned = TIME_RE.sub("", cleaned)
        cleaned = DAY_RE.sub("", cleaned)
        cleaned = re.sub(r"^[\W_]+|[\W_]+$", "", cleaned).strip()
        if len(cleaned) > 2:
            addr_parts.append(cleaned)

    raw_addr = ", ".join(p.rstrip(", ") for p in addr_parts)
    clean_addr = re.sub(r"([.,])(?=[A-Za-z0-9])", r"\1 ", raw_addr)
    clean_addr = re.sub(r"(\d{5})(?=[A-Za-z])", r"\1 ", clean_addr)
    for pat, rep in [
        (r"\bJln\.?\b", "Jalan"),
        (r"\bTmn\.?\b", "Taman"),
        (r"\bSek\.?\b", "Seksyen"),
        (r"\bKg\.?\b", "Kampung"),
        (r"\bBdr\.?\b", "Bandar"),
    ]:
        clean_addr = re.sub(pat, rep, clean_addr, flags=re.IGNORECASE)
    clean_addr = re.sub(r"\s+", " ", clean_addr).strip()

    # 1. Split fused CamelCase words (e.g., "SeriKembangan" -> "Seri Kembangan")
    clean_addr = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", clean_addr)
    # 2. Remove false commas after street/area prefixes caused by line wraps ("Jalan, Medan" -> "Jalan Medan")
    clean_addr = re.sub(
        r"\b(Jalan|Lorong|Persiaran|Taman|Seksyen|Bandar|Kampung),\s*",
        r"\1 ",
        clean_addr,
        flags=re.IGNORECASE,
    )
    # Strip floor/lot/unit prefix only when preceded by No./Lot/Unit/Blok or alphanumeric unit code (e.g., "5-A")
    cluster_key = re.sub(
        r"^(?:(?:Ground|1st|2nd|3rd|\d+th)\s+Floor,\s*)?"
        r"(?:(?:No|Lot|Unit|Blok|Block)\.?\s*[A-Za-z0-9-]+|(?:[A-Za-z]*\d+[A-Za-z0-9-]*))[,\s]+\s*",
        "",
        clean_addr,
        flags=re.IGNORECASE,
    )

    return {
        "display_date": ts["display_date"],
        "display_time": ts["display_time"],
        "captured_at_iso": ts["captured_at_iso"],
        "postcode": postcode,
        "gps": gps_coords,
        "full_address": clean_addr,
        "cluster_key": cluster_key,
    }


# 5. Dedicated Layout Extractors (Strategy Pattern)
def extract_layout_a(img) -> dict:
    """Layout A: Timemark Classic (Bottom-Left Split Clock + MBSJ Emblem)"""
    h, w, _ = img.shape
    boxes = preprocess_and_ocr(img[int(h * 0.66):h, 0:int(w * 0.62)], darken_concrete=False)
    anchors = [b for b in boxes if DATE_RE.search(b["text"]) or TIME_RE.search(b["text"])]
    if anchors:
        y_min = min(a["y_top"] for a in anchors)
        boxes = [b for b in boxes if b["y_bottom"] >= y_min - 20]
    return finalize_fields([b["text"] for b in boxes])


def extract_layout_b(img, gps_coords: str) -> dict:
    """Layout B: Timemark GPS Malay Locale (Portrait + Top GPS Pill)"""
    h, w, _ = img.shape
    boxes = preprocess_and_ocr(img[int(h * 0.78):h, 0:int(w * 0.76)], darken_concrete=True)
    anchors = [b for b in boxes if DATE_RE.search(b["text"]) or TIME_RE.search(b["text"])]
    if anchors:
        y_min = min(a["y_top"] for a in anchors)
        boxes = [b for b in boxes if b["y_bottom"] >= y_min - 20]
    return finalize_fields([b["text"] for b in boxes], gps_coords=gps_coords)


def extract_layout_c(img) -> dict:
    """Layout C: Timestamp Camera (Bottom-Right White Text Block)"""
    h, w, _ = img.shape
    boxes = preprocess_and_ocr(img[int(h * 0.72):h, int(w * 0.48):w], darken_concrete=True)
    anchors = [b for b in boxes if DATE_RE.search(b["text"]) or TIME_RE.search(b["text"])]
    if anchors:
        y_min = min(a["y_top"] for a in anchors)
        boxes = [b for b in boxes if b["y_bottom"] >= y_min - 20]
    return finalize_fields([b["text"] for b in boxes])


# 6. Automatic Layout Router
def route_and_extract(image_path: str) -> tuple[str, dict]:
    img = cv2.imread(image_path)
    h, w, _ = img.shape

    top_left_boxes = preprocess_and_ocr(img[0:int(h * 0.22), 0:int(w * 0.65)])
    gps_m = GPS_RE.search(" ".join(b["text"] for b in top_left_boxes))
    if gps_m or h > w:
        gps_str = (
            f"{gps_m.group(1)}°{gps_m.group(2).upper()}, {gps_m.group(3)}°{gps_m.group(4).upper()}"
            if gps_m else "NOT PRESENT"
        )
        return "Layout B (Timemark GPS Malay)", extract_layout_b(img, gps_str)

    res_a = extract_layout_a(img)
    if res_a["display_date"] != "NOT FOUND" or res_a["display_time"] != "NOT FOUND":
        return "Layout A (Timemark Classic)", res_a

    return "Layout C (Timestamp Camera Right)", extract_layout_c(img)


# 7. Batch Run on Uploaded Sample Images
uploaded = files.upload()
for path in uploaded.keys():
    layout_name, data = route_and_extract(path)
    print(f"\n================ {path} -> [{layout_name}] ================")
    for k, v in data.items():
        print(f"{k:<16}: {v}")

Saving WhatsApp Image 2026-01-15 at 07.01.37 (1).jpeg to WhatsApp Image 2026-01-15 at 07.01.37 (1) (1).jpeg
Saving WhatsApp Image 2026-01-15 at 07.19.32.jpeg to WhatsApp Image 2026-01-15 at 07.19.32 (1).jpeg
Saving WhatsApp Image 2026-01-15 at 07.20.01 (2).jpeg to WhatsApp Image 2026-01-15 at 07.20.01 (2) (1).jpeg

================ WhatsApp Image 2026-01-15 at 07.01.37 (1) (1).jpeg -> [Layout B (Timemark GPS Malay)] ================
display_date    : 12/01/2026
display_time    : 08:48:00
captured_at_iso : 2026-01-12T08:48:00+08:00
postcode        : 43300
gps             : 3.023318°N, 101.705778°E
full_address    : 1st Floor, 5-A, Jalan Besar, Kampung Baru, Seri Kembangan, 43300 Seri Kembangan, Selangor
cluster_key     : Jalan Besar, Kampung Baru, Seri Kembangan, 43300 Seri Kembangan, Selangor

================ WhatsApp Image 2026-01-15 at 07.19.32 (1).jpeg -> [Layout C (Timestamp Camera Right)] ================
display_date    : 09/01/2026
display_time    : 08:57:53
captured_at_iso : 2